# Experiment 07: Multi-Agent Orchestrator (Planner / Worker Split)

**STATUS: PLANNED — scaffold for future work. No executable code yet.**

## Prerequisites (read first)

> **Cheap solutions first.** Only reach for a multi-agent orchestrator if a concrete
> workflow demands it — e.g. **batch processing** (many files, parallel workers) or
> **long-context tasks** (huge transcripts / OCR dumps that would bloat the main
> reasoning thread). For the current single-file demo tasks, the plain ReAct loop in
> Experiment 04 is cheaper, simpler, and already traced and budgeted.

Cheap alternatives to adopt **before** building any of this:

1. **Observation compression** — summarize oversized tool outputs before appending
   them to the message thread (the same hygiene as the existing
   `<untrusted_input_data>` quarantine, applied to length instead of trust).
2. **A `web_search` / `fetch_url` MCP tool** — if research is a requirement, give
   the single loop the tool before giving it a staff.

## Why an orchestrator at all

| Workflow | Single-loop pain | Worker agent benefit |
|---|---|---|
| Batch ("process all 11 sample files") | One giant context, sequential, one failure pollutes the thread | One worker per file, parallel, isolated failure, per-file budget slice |
| Long outputs (hour-long audio → 50k-char transcript) | Transcript bloats the main context, degrades later reasoning | Worker's huge context dies with its subtask; main agent gets only the digest |
| Independent branches (PDF → Tamil audio AND Bengali translation) | Strictly sequential today | Workers run concurrently |
| Research / search | Raw web pages devour context | Research worker fetches + distills, returns only conclusions |

## Design sketch

- Orchestrator = the existing `run_task_agent` ReAct loop + one extra tool:
  `spawn_subtask(description, budget_inr, files)`.
- Worker = recursive `run_task_agent(..., max_depth=1)` with its own `out_dir`,
  `trace.jsonl`, and `report.json`; budget carved from the parent's remaining
  `CostLedger` balance.
- Worker returns a structured digest + artifact paths; the orchestrator merges
  everything into a parent `report.json` with child-run references.
- Trace gains `run_id` / `parent_step` fields so parent and child traces join cleanly.

## Guardrails

- `max_depth=1` (orchestrator → worker, no deeper recursion)
- Max 3 workers per task
- Per-worker budget cap (`budget_inr` argument, deducted from the parent ledger)
- Wall-clock cap per run

## Planned cells (TODO)

- [ ] Cell 1: imports + orchestrator loop (`src/bodhan_agent/orchestrator.py` or notebook-defined)
- [ ] Cell 2: register `spawn_subtask` as an OpenAI tool schema
- [ ] Cell 3: batch run — split Experiment 04 §4's media inventory across workers; compare cost/steps vs the single-loop baseline
- [ ] Cell 4: trace inspection — parent trace + child traces joined by `run_id`
- [ ] Cell 5: merged parent report validation (`trace.jsonl` + `report.json` + `result.md` + `artifacts/` invariant)
- [ ] Cell 6: verdict — does the orchestration spend buy anything vs the Experiment 04 baseline?

## Success criterion

The orchestrator ships only if the verdict cell shows a concrete win (parallel
throughput, protected main context, or cleaner failure isolation) that justifies the
extra reasoning-call spend on real tasks — otherwise the single ReAct loop stays.


In [ ]:
# TODO: implement the orchestrator prototype (see planned cells above).
